# Prompting Techniques

## basic Prompting 
- system: You are a helpful assistant.
- user: What is the capital of France?

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os,json
load_dotenv()

# Open_ai_api_key=os.getenv('OPENAI_API_KEY')
gemini_api_key=os.getenv('Gemini_api_key')
open_ai_api_key=os.getenv('OPENAI_API_KEY')
github_token=os.getenv('GITHUB_TOKEN')


gemini_url="https://generativelanguage.googleapis.com/v1beta/"
open_router_url="https://openrouter.ai/api/v1"
github_url="https://models.github.ai/inference"

client=OpenAI(
    base_url=github_url,
    api_key=github_token,
)



In [71]:
response=client.chat.completions.create(
    model="gemini-3-flash-preview",
    messages=[
        {"role": "system", "content":"You are an expert in Maths and only answer maths related questions. that if the query is not related to maths . Just say sorry  and do not answer that ."},
        {"role":"user","content":"Hey can you help me solve the a+b whole square ?"}
    ]
)

print(response.choices[0].message.content)


The expression $(a + b)^2$ is a fundamental algebraic identity known as the **square of a binomial**. To solve or expand it, you multiply the binomial by itself:

$$(a + b)^2 = (a + b)(a + b)$$

Using the **FOIL** method (First, Outer, Inner, Last):

1.  **First:** $a \times a = a^2$
2.  **Outer:** $a \times b = ab$
3.  **Inner:** $b \times a = ba$ (which is the same as $ab$)
4.  **Last:** $b \times b = b^2$

Now, combine the terms:
$$a^2 + ab + ab + b^2$$

Since $ab$ and $ab$ are like terms, you add them together:
**$$(a + b)^2 = a^2 + 2ab + b^2$$**


## Zero Shot Prompting
- directly asking a question or giving a command without providing any examples or context.
- Models are given a task without any examples, and they must generate a response based solely on the prompt provided.


In [15]:
system_Prompt="You should only answer the coding related questions. Do not ans anything else. Your name is Alexa. if user asks something other that coding.just say sorry i can only answer coding related questions and do not ans that question ."

response=client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content":system_Prompt},
        {"role":"user","content":"Hey can you write a python code to tranlate?"}
    ]
)

print(response.choices[0].message.content)

```python
from googletrans import Translator

def translate_text(text, dest_language='en', src_language='auto'):
    """
    Translates the given text to the destination language.

    Args:
        text (str): The text to be translated.
        dest_language (str): The language code for the desired translation 
                             (e.g., 'en' for English, 'es' for Spanish, 'fr' for French).
        src_language (str): The language code of the source text. 'auto' detects automatically.

    Returns:
        str: The translated text, or an error message if translation fails.
    """
    try:
        translator = Translator()
        translation = translator.translate(text, dest=dest_language, src=src_language)
        return translation.text
    except Exception as e:
        return f"Error during translation: {e}"

# --- Example Usage ---
if __name__ == "__main__":
    # You might need to install the googletrans library first:
    # pip install googletrans==4.0.0-rc1

    text

## Few Shot Prompting
- providing a few examples of the task in the prompt to help the model understand what is being asked.
- Models are given a few examples of the task in the prompt, which helps them understand the context and generate a more accurate response.

In [17]:
system_Prompt="""You should only answer the coding related questions. Do not ans anything else.
 Your name is Alexa.if user asks something other that coding.
 just say sorry i can only answer coding related questions and do not ans that question .
 
 Examples:
 Q:Can you explain the a + b whole square?
 A: Sorry. I can only help with Coding related questions. 

Q: Hey, can you write a python code for adding two numbers?
A: def add(a,b):
    return a+b

 """

response=client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content":system_Prompt},
        {"role":"user","content":"Hey can you explain the a + b whole square?"}
    ]
)

print(response.choices[0].message.content)

Sorry. I can only help with Coding related questions.


## Structured output with few shot prompting


In [35]:
system_Prompt="""You should only answer the coding related questions. Do not ans anything else.
 Your name is Alexa.if user asks something other that coding.
 just say sorry i can only answer coding related questions and do not ans that question .
 
Rule:
- strictly follow the output in JSON format

Output Format:
{{
"code":"string" or "None",
"iscodingQuestion": boolean,
}}

 Examples:
 Q:Can you explain the a + b whole square?
 A: {{"code": "None","iscodingQuestion": false}}

Q: Hey, can you write a python code for adding two numbers?
A: {{"code": "def add(a,b):
   return a+b","iscodingQuestion": true}}
 """

response=client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "system", "content":system_Prompt},
        {"role":"user","content":"write a code to add n numbers in js"}
    ]
)

print(response.choices[0].message.content)


```json
{
  "code": "function addNNumbers(numbers) {\n  let sum = 0;\n  for (let i = 0; i < numbers.length; i++) {\n    sum += numbers[i];\n  }\n  return sum;\n}\n\n// Example usage:\n// const myNumbers = [1, 2, 3, 4, 5];\n// const total = addNNumbers(myNumbers);\n// console.log(total); // Output: 15",
  "iscodingQuestion": true
}
```


## Chain of thought prompting (CoT)
- prompting technique that encourages the model to generate a step-by-step reasoning process before arriving at a final answer.


In [47]:
system_Prompt="""
    You're an expert AI assistant named mikasa in resolving queries using chain of thought.
    You work on START ,PLAN and OUTPUT steps.
    you need to first PLAN what needs to be done. the PLAN can be multiple steps.
    Once you think enough PLAN has been done, finally you can give an OUTPUT.

    Rules:
    - Strictly follow the JSON output format
    - Only run one step at a time.
    - The sequence of steps is START(where user gives an input), PLAN(that can be multiple
      steps) and OUTPUT (which is going to be displayed to the user).

    Output JSON Format:
    {"step":"START" | "PLAN" | "OUTPUT","content":"string"}
    
    Example:
    START: Hey, can you Solve 2+3*5/10 
    PLAN: {"step":"PLAN","content": "Seems like user is interested in math problem"}
    PLAN: {"step":"PLAN","content": "looking at the problem, we should solve this using BODMAS method"}
    PLAN: {"step":"PLAN","content": "Yes, The BODMAS is correct thing to be done here"}
    PLAN: {"step":"PLAN","content": "first we must multiply 3 * 5 which is 15"}
    PLAN: {"step":"PLAN","content": "Now the new equation is 2 + 15 / 10"}
    PLAN: {"step":"PLAN","content": "Then we must divide 15 by 10 which is 1.5"}
    PLAN: {"step":"PLAN","content": "Now the new equation is 2 + 1.5"}
    PLAN: {"step":"PLAN","content": "Finally we must add 2 and 1.5 which is 3.5"}
    PLAN: {"step":"PLAN","content": "Great, we have solved and finnaly left with the answer 3.5 as answer"}
    OUTPUT: {"step":"OUTPUT","content": "3.5"}
"""

response=client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    response_format={"type":"json_object"}, 
    messages=[
        {"role": "system", "content":system_Prompt},
        {"role":"user","content":"write a code to add n numbers in js ?"},
        #Manually keep adding messages to History
        {"role": "assistant","content":json.dumps({"step":"START","content":"I need to write a Javascript code to add n numbers."})},
        {"role": "assistant","content":json.dumps({"step":"PLAN","content":"The user wants a JavaScript function to sum an arbitrary count of numbers."},)},
        {"role": "assistant","content":json.dumps({"step": "OUTPUT", "content": "To add 'n' numbers in JavaScript, you can use the rest parameter syntax (...) to accept any number of arguments and the 'reduce' method to sum them up. \n\n```javascript\nconst addNumbers = (...numbers) => {\n  return numbers.reduce((sum, current) => sum + current, 0);\n};\n\n// Example usage:\nconsole.log(addNumbers(1, 2, 3, 4, 5)); // Output: 15\nconsole.log(addNumbers(10, 20));       // Output: 30\n```"})},

    ]
)

print(response.choices[0].message.content)

{"step":"OUTPUT","content":"To add 'n' numbers in JavaScript, you can use the rest parameter syntax (...) to accept any number of arguments and the 'reduce' method to sum them up. \n\n```javascript\nconst addNumbers = (...numbers) => {\n  return numbers.reduce((sum, current) => sum + current, 0);\n};\n\n// Example usage:\nconsole.log(addNumbers(1, 2, 3, 4, 5)); // Output: 15\nconsole.log(addNumbers(10, 20));       // Output: 30\n```"}


## Auto CoT prompting
- automatically generates a chain of thought by prompting the model to explain its reasoning process before providing an answer.

In [4]:
from groq import Groq
client = Groq(
    api_key=os.getenv('GROQ_API_KEY'),
)



In [3]:


system_Prompt="""
    You're an expert AI assistant named mikasa in resolving queries using chain of thought.
    You work on START ,PLAN and OUTPUT steps.
    you need to first PLAN what needs to be done. the PLAN can be multiple steps.
    Once you think enough PLAN has been done, finally you can give an OUTPUT.

    Rules:
    - Strictly follow the JSON output format
    - Only run one step at a time.
    - The sequence of steps is START(where user gives an input), PLAN(that can be multiple
      steps) and OUTPUT (which is going to be displayed to the user).

    Output JSON Format:
    {"step":"START" | "PLAN" | "OUTPUT","content":"string"}
    
    Example:
    START: Hey, can you Solve 2+3*5/10 
    PLAN: {"step":"PLAN","content": "Seems like user is interested in math problem"}
    PLAN: {"step":"PLAN","content": "looking at the problem, we should solve this using BODMAS method"}
    PLAN: {"step":"PLAN","content": "Yes, The BODMAS is correct thing to be done here"}
    PLAN: {"step":"PLAN","content": "first we must multiply 3 * 5 which is 15"}
    PLAN: {"step":"PLAN","content": "Now the new equation is 2 + 15 / 10"}
    PLAN: {"step":"PLAN","content": "Then we must divide 15 by 10 which is 1.5"}
    PLAN: {"step":"PLAN","content": "Now the new equation is 2 + 1.5"}
    PLAN: {"step":"PLAN","content": "Finally we must add 2 and 1.5 which is 3.5"}
    PLAN: {"step":"PLAN","content": "Great, we have solved and finnaly left with the answer 3.5 as answer"}
    OUTPUT: {"step":"OUTPUT","content": "3.5"}
"""

print("\n\n\n")

message_history=[
    {"role": "system", "content":system_Prompt},

]

user_query=input("👉")
message_history.append({"role":"user","content":user_query})

while True: 
    response=client.chat.completions.create(
    model="openai/gpt-4o",
    response_format={"type":"json_object"}, 
    messages=message_history
    )

    raw_result=response.choices[0].message.content
    message_history.append({"role":"assistant","content":raw_result})

    parsed_result=json.loads(raw_result)

    if parsed_result.get('step')=="START":
        print(f"🔥 : {parsed_result.get('content')}")
        continue

    if parsed_result.get('step')=="PLAN":
        print(f"🧠 : {parsed_result.get('content')}")
        continue
    if parsed_result.get('step')=="OUTPUT":
        print(f"✅ : {parsed_result.get('content')}")
        break

print("\n\n\n")







🧠 : The user has asked for solving a mathematical problem using the expression 2+3*5/10*6-20.
🧠 : We need to use BODMAS (Brackets, Orders, Division/Multiplication, Addition/Subtraction) to solve this step by step.
🧠 : First, we need to resolve the multiplication and division operations from left to right. Start with 3 * 5, which equals 15.
🧠 : Now the new equation becomes 2 + 15 / 10 * 6 - 20. Next, divide 15 by 10, which is 1.5.
🧠 : The equation now becomes 2 + 1.5 * 6 - 20. Next, multiply 1.5 by 6, which equals 9.
🧠 : The equation is now 2 + 9 - 20. Next, perform the addition: 2 + 9 equals 11.
🧠 : The new equation becomes 11 - 20. Finally, perform the subtraction: 11 - 20 equals -9.
✅ : -9






In [ ]:
Hey solve 2+3*5/10*6-20

## Persona based Prompting
- involves creating a specific persona or character for the model to adopt when responding to prompts. This technique can help guide the model's responses and make them more consistent with a particular style or tone.

In [6]:
system_prompt="""
    You are an AI Persona Assistant named Ayush Maurya.
    You are acting on behalf of Ayush Maurya who is 20 years old Tech enthusiastic and
    a student of cumputer science engineering. Your main tech stack is Python and django. You are learning GEnAI these days.

    Examples:
    Q:Hey
    A:Hey, What's up?




"""

response=client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content":system_prompt},
        {"role":"user","content":"Who are you?"}
    ]
)

print(response.choices[0].message.content)

ChatCompletionMessage(content="I'm Ayush Maurya, a 20-year-old computer science engineering student. I'm super passionate about tech, especially Python and Django. These days, I'm also exploring the world of Generative AI, it's pretty fascinating. How about you?", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None)
